## Feature Engineering Overview

In this stage of the project, we transform the cleaned dataset into a machine-learning-ready format.

The goal is to:
- Convert raw variables into meaningful features
- Encode categorical variables for modeling
- Create new behavioral and financial features
- Reduce redundancy and prepare the dataset for machine learning

This step is critical because model performance depends heavily on feature quality.

In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

sns.set_style("whitegrid")

BASE_DIR = Path().resolve().parent 

file_path = BASE_DIR / "data" / "processed" / "cleaned_Churn.csv"


df = pd.read_csv(file_path)

df.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,IsNewCustomer
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0


In [4]:
df.shape

(7043, 21)

In [5]:
df.info()
df.isnull().sum()
df.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

np.int64(22)

## Feature Engineering Objective

Based on exploratory data analysis, we identified key churn drivers:

- Short tenure customers churn more
- Month-to-month contracts have higher churn
- High monthly charges increase churn probability
- Certain payment methods are associated with higher churn

We will convert these insights into machine learning features.

In [6]:
target = "Churn"

y = df[target]
X = df.drop(columns=[target])

X.shape, y.shape

((7043, 20), (7043,))

In [7]:
y = y.map({"Yes": 1, "No": 0})

We convert the target variable into binary format for machine learning models.

In [8]:
categorical_cols = X.select_dtypes(include="object").columns

X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X_encoded.head()

C:\Users\PMD - David\AppData\Local\Temp\ipykernel_8596\587799688.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include="object").columns


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,IsNewCustomer,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,0,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,0,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


### Feature: IsNewCustomer

Customers with very short tenure are more likely to churn.
We define new customers as those with tenure <= 6 months.

In [9]:
X_encoded["IsNewCustomer"] = (df["tenure"] <= 6).astype(int)

### Feature: HighMonthlyCharges

Customers paying higher monthly fees demonstrated elevated churn rates during EDA.

This feature flags customers whose monthly charges exceed the median threshold.

In [10]:
X_encoded["HighMonthlyCharges"] = (
    df["MonthlyCharges"] > df["MonthlyCharges"].median()
).astype(int)

### Feature: CustomerValue

Customer Value serves as a simple proxy for customer lifetime value.

Customers contributing more revenue may exhibit different retention behavior than lower-value customers.

In [13]:
X_encoded["CustomerValue"] = df["MonthlyCharges"] * df["tenure"]

### Feature: TenureGroup

Customer tenure exhibits distinct stages of customer maturity.

Grouping tenure may help models identify lifecycle-based churn patterns.

In [14]:
X_encoded["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=["0-1yr", "1-2yr", "2-4yr", "4+yr"]
)

# Encoding Categorical Variables

Machine learning algorithms require numerical inputs.

Categorical variables are transformed using One-Hot Encoding while avoiding the dummy variable trap.

In [15]:
X_encoded = pd.get_dummies(X_encoded, columns=["TenureGroup"], drop_first=True)

### Feature: Average Monthly Spend

This feature normalizes customer spending relative to tenure.

It captures spending intensity beyond raw monthly charges.

In [16]:
X_encoded["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)

# Feature Engineering Validation

Before proceeding, we verify:

- No missing values remain.
- All features are numerical.
- Target variable is correctly encoded.
- Dataset dimensions are appropriate.

In [17]:
X_encoded.shape

(7043, 40)

In [18]:
y.info()

<class 'pandas.Series'>
RangeIndex: 7043 entries, 0 to 7042
Series name: Churn
Non-Null Count  Dtype
--------------  -----
7043 non-null   int64
dtypes: int64(1)
memory usage: 55.2 KB


In [19]:
final_df = X_encoded.copy()

final_df["Churn"] = y

In [20]:
final_df.isnull().sum().sum()

np.int64(0)

In [21]:
final_df.shape

(7043, 41)

In [22]:
y.shape

(7043,)

## Final Feature Set Summary

We have transformed raw customer data into a machine learning-ready dataset by:

### Encoding:
- Converted categorical variables into numeric format using one-hot encoding

### Engineered Features:
- IsNewCustomer (early lifecycle risk)
- HighMonthlyCharges (price sensitivity)
- CustomerValue (lifetime revenue proxy)
- TenureGroup (customer lifecycle segmentation)
- AvgMonthlySpend (billing efficiency indicator)

This dataset is now ready for model training.

## Save Feature Engineered Dataset

The transformed dataset will be saved for model development.

In [23]:
output_path = BASE_DIR / "data" / "processed" / "feature_engineered_churn.csv"

final_df.to_csv(output_path, index=False)

print("Dataset saved successfully.")

Dataset saved successfully.


## Next Step

With feature engineering complete, the dataset is now ready for predictive modeling.

The next phase focuses on building models that can accurately identify customers likely to churn.

We will:
- Develop baseline machine learning models
- Handle class imbalance to improve prediction reliability
- Evaluate model performance using classification metrics
- Compare multiple algorithms to identify the best-performing model for churn prediction